# E-commerce Marketplace Financial Analysis - Multi Country

Analyzing sales data and payment status from marketplace across multiple countries

In [1]:
import pandas as pd
import numpy as np
import glob
import os

# Available countries
countries = ['de', 'fr', 'it', 'at', 'cz', 'pl', 'sk']

print("Available CSV files:")
for country in countries:
    gmu_files = glob.glob(f'report_booking_gmu_{country}_*.csv')
    df10k_files = glob.glob(f'report_booking10000_{country}_*.csv')
    print(f"  {country.upper()}: GMU={len(gmu_files)}, df_10000={len(df10k_files)}")

Available CSV files:
  DE: GMU=1, df_10000=1
  FR: GMU=1, df_10000=1
  IT: GMU=1, df_10000=1
  AT: GMU=1, df_10000=1
  CZ: GMU=1, df_10000=1
  PL: GMU=1, df_10000=1
  SK: GMU=1, df_10000=1


In [2]:
# SELECT COUNTRY HERE
COUNTRY = 'at'  # Change this to: 'de', 'fr', 'it', 'at', 'cz', 'pl', or 'sk'

# Find the CSV files for this country
gmu_file = glob.glob(f'report_booking_gmu_{COUNTRY}_*.csv')[0]
df10k_file = glob.glob(f'report_booking10000_{COUNTRY}_*.csv')[0]

print(f"\n{'='*70}")
print(f"ANALYZING COUNTRY: {COUNTRY.upper()}")
print(f"{'='*70}")
print(f"\nGMU file: {os.path.basename(gmu_file)}")
print(f"df_10000 file: {os.path.basename(df10k_file)}")

# Load the CSV files with semicolon delimiter and handle German number format
df_gmu = pd.read_csv(
    gmu_file,
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

df_10000 = pd.read_csv(
    df10k_file,
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

# Convert dates to datetime
df_gmu['booking_date'] = pd.to_datetime(df_gmu['booking_date'])
df_10000['Datum'] = pd.to_datetime(df_10000['Datum'])

# Add month column
df_gmu['month'] = df_gmu['booking_date'].dt.to_period('M')
df_10000['month'] = df_10000['Datum'].dt.to_period('M')

print(f"\n✓ Datasets loaded successfully")
print(f"GMU: {df_gmu.shape[0]} rows, {df_gmu.shape[1]} columns")
print(f"  Date range: {df_gmu['booking_date'].min()} to {df_gmu['booking_date'].max()}")
print(f"df_10000: {df_10000.shape[0]} rows, {df_10000.shape[1]} columns")
print(f"  Date range: {df_10000['Datum'].min()} to {df_10000['Datum'].max()}")


ANALYZING COUNTRY: AT

GMU file: report_booking_gmu_at_a099b98f43e83968998863f7cd08b58bc0cd648d5638a2cac4d87f5072543216.csv
df_10000 file: report_booking10000_at_aac1b30c32f25a99bf02b728312b644169492b6bf7c324b0140c5f77a5fb97a4.csv

✓ Datasets loaded successfully
GMU: 42 rows, 54 columns
  Date range: 2025-08-07 11:50:23 to 2025-10-27 08:37:44
df_10000: 98 rows, 9 columns
  Date range: 2025-07-17 00:00:00 to 2025-10-26 00:00:00


In [3]:
# Understanding the data structure
print("="*70)
print("UNDERSTANDING THE DATA STRUCTURE")
print("="*70)

print("\n1. GMU Dataset - Transaction Types:")
print(df_gmu['booking_text'].value_counts().head(10))

print("\n2. df_10000 Dataset - Transaction Types:")
print(df_10000['Buchungstext'].value_counts().head(10))

print("\n3. Understanding Payment Flow (multi-language):")
print("  - Wareneingang/Goods receipt/Entrata merce (Sales Income): Order placed, positive amount")
print("  - Netto Provision/Net commission/Commissione netta (Commission): Commission charged, negative amount")
print("  - Freigabe/Release/Rilascio (Sales Released): PAYMENT RECEIVED, negative amount")
print("  - An order is PAID when 'Freigabe/Release/Rilascio' transaction exists in df_10000")

UNDERSTANDING THE DATA STRUCTURE

1. GMU Dataset - Transaction Types:
booking_text
Payout                                                                                                                16
Sponsored Product Ads - click costs; billing period 2025-07-01 - 2025-07-31; Number of Clicks: 45                      1
Release order MCEZUFQ/314567988348649 InnovaGoods Elektrische Wärmflasche                                              1
Release order M3U69LQ/314567988992253 Brazier FM Gruppe BL1450 450 W                                                   1
Release order M44B8FQ/314567988626335 Phoenix Technologies 135 ́ ́ Projektionsfläche 2.4x2.4 M Silber One Size ...     1
Release order M4XPEFQ/314567988356046 loft urban rund rollen 50cm theaterrot                                           1
Release order MUF69LQ/314567988992397 Pandora Funkelnde Unendlichkeits-Herz Collier-Halskette 392666C01-50 Dame...     1
Release order MHD71LQ/314567988842879 BRIGHT STARTS - Minnie Mouse vib

In [ ]:
# Multi-language keyword patterns
# These patterns work across all languages
RELEASE_KEYWORDS = ['Freigabe', 'Release', 'Rilascio', 'Libération', 'Zwolnienie', 'Uvolnění', 'Uvoľnenie']
GOODS_RECEIPT_KEYWORDS = ['Wareneingang', 'Goods receipt', 'Entrata merce', 'Réception', 'Przyjęcie', 'Příjem', 'Príjem']
COMMISSION_KEYWORDS = ['Provision', 'commission', 'Commissione', 'Commission', 'Prowizja', 'Provize']
PAYOUT_KEYWORDS = ['Payout', 'Auszahlung', 'Pagamento', 'Paiement', 'Wypłata', 'Výplata']
BASE_FEE_KEYWORDS = ['Grundgebühr', 'Base fee', 'Tariffa base', 'Frais de base', 'Opłata podstawowa', 'Základní poplatek']
CANCEL_KEYWORDS = ['Cancel', 'Storno', 'Annul', 'Anulacja', 'Annulation']

def contains_any(text, keywords):
    """Check if text contains any of the keywords (case insensitive)"""
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

# Identify order status: PAID, UNPAID, or CANCELLED
print("="*70)
print("IDENTIFYING ORDER STATUS")
print("="*70)

# Get unique order numbers from both datasets
gmu_orders = set(df_gmu['order_number'].dropna().unique())
df10000_all_orders = set(df_10000['Bestellnummer'].dropna().unique())

# Orders are PAID only if they have a 'Freigabe/Release' (Sales Released) transaction
freigabe_orders = set(
    df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, RELEASE_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

# Orders with Wareneingang/Goods receipt (created) in df_10000
wareneingang_orders = set(
    df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, GOODS_RECEIPT_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

# CANCELLED: Orders with "Cancel" in Buchungstext
cancelled_orders = set(
    df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, CANCEL_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

print(f"\nUnique orders in GMU: {len(gmu_orders)}")
print(f"Unique orders in df_10000: {len(df10000_all_orders)}")
print(f"Orders with Wareneingang (created): {len(wareneingang_orders)}")
print(f"Orders with Freigabe/Release (PAID): {len(freigabe_orders)}")
print(f"Orders with Cancel (CANCELLED): {len(cancelled_orders)}")

# UNPAID = Created but not paid AND not cancelled
unpaid_orders_set = (wareneingang_orders - freigabe_orders) - cancelled_orders

print(f"\nPAID orders: {len(freigabe_orders)}")
print(f"UNPAID orders (created, not paid, not cancelled): {len(unpaid_orders_set)}")
print(f"CANCELLED orders: {len(cancelled_orders)}")

if len(unpaid_orders_set) > 0:
    print(f"\nSample unpaid order numbers: {list(unpaid_orders_set)[:5]}")
if len(cancelled_orders) > 0:
    print(f"Sample cancelled order numbers: {list(cancelled_orders)[:5]}")

In [ ]:
# Create comprehensive order list from BOTH sources
print("="*70)
print("BUILDING COMPLETE ORDER LIST")
print("="*70)

# Start with GMU sales
gmu_sales = df_gmu[df_gmu['order_number'].notna()].copy()
gmu_sales['is_paid'] = gmu_sales['order_number'].isin(freigabe_orders)
gmu_sales['is_cancelled'] = gmu_sales['order_number'].isin(cancelled_orders)
gmu_sales['source'] = 'GMU'

# Add order_status field
gmu_sales['order_status'] = 'UNKNOWN'
gmu_sales.loc[gmu_sales['is_paid'], 'order_status'] = 'PAID'
gmu_sales.loc[gmu_sales['is_cancelled'], 'order_status'] = 'CANCELLED'
gmu_sales.loc[(~gmu_sales['is_paid']) & (~gmu_sales['is_cancelled']), 'order_status'] = 'UNPAID'

print(f"\nOrders from GMU: {len(gmu_sales)}")
print(f"  Paid: {gmu_sales['is_paid'].sum()}")
print(f"  Cancelled: {gmu_sales['is_cancelled'].sum()}")
print(f"  Unpaid: {((~gmu_sales['is_paid']) & (~gmu_sales['is_cancelled'])).sum()}")

# Find orders in df_10000 that are NOT in GMU (recent orders)
orders_only_in_df10000 = wareneingang_orders - gmu_orders
print(f"\nOrders ONLY in df_10000 (not yet in GMU): {len(orders_only_in_df10000)}")

if len(orders_only_in_df10000) > 0:
    print(f"  These are recent orders: {list(orders_only_in_df10000)[:5]}")
    
    # Create entries for these orders from df_10000 data
    wareneingang_df = df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, GOODS_RECEIPT_KEYWORDS))].copy()
    commission_df = df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, COMMISSION_KEYWORDS))].copy()
    
    recent_orders_list = []
    for order_num in orders_only_in_df10000:
        waren_row = wareneingang_df[wareneingang_df['Bestellnummer'] == order_num]
        comm_row = commission_df[commission_df['Bestellnummer'] == order_num]
        
        if len(waren_row) > 0:
            row = waren_row.iloc[0]
            commission = abs(comm_row.iloc[0]['Betrag']) if len(comm_row) > 0 else 0
            gross = row['Betrag']
            
            is_paid = order_num in freigabe_orders
            is_cancelled = order_num in cancelled_orders
            
            if is_paid:
                order_status = 'PAID'
            elif is_cancelled:
                order_status = 'CANCELLED'
            else:
                order_status = 'UNPAID'
            
            recent_orders_list.append({
                'order_number': order_num,
                'booking_date': row['Datum'],
                'order_date': row['Datum'],
                'booking_text': row['Buchungstext'],
                'price_gross': gross,
                'sum_price_gross': gross,
                'fee_gross': commission,
                'payout': gross - commission,
                'is_paid': is_paid,
                'is_cancelled': is_cancelled,
                'order_status': order_status,
                'source': 'df_10000_only',
                'title_item': 'See df_10000 for details',
                'shipping_charges_gross': 0,
                'fee_%': (commission / gross * 100) if gross > 0 else 0,
                'buyer.email': '',
                'shipping.first_name': '',
                'shipping.last_name': '',
                'shipping.city': ''
            })
    
    recent_orders_df = pd.DataFrame(recent_orders_list)
    print(f"  Created {len(recent_orders_df)} entries from df_10000")
    
    # Combine with GMU sales
    all_orders = pd.concat([gmu_sales, recent_orders_df], ignore_index=True)
else:
    all_orders = gmu_sales

# Final counts
paid_orders = all_orders[all_orders['order_status'] == 'PAID'].copy()
unpaid_orders = all_orders[all_orders['order_status'] == 'UNPAID'].copy()
cancelled_orders_list = all_orders[all_orders['order_status'] == 'CANCELLED'].copy()

print(f"\nFINAL COUNTS:")
print(f"  Total orders: {len(all_orders)}")
print(f"  Paid: {len(paid_orders)}")
print(f"  Unpaid: {len(unpaid_orders)}")
print(f"  Cancelled: {len(cancelled_orders_list)}")

In [ ]:
# Comprehensive financial summary
print("="*70)
print("FINANCIAL SUMMARY")
print("="*70)

summary = {
    'Total Sales': {
        'Gross Sales': all_orders['price_gross'].sum(),
        'Shipping': all_orders['shipping_charges_gross'].sum(),
        'Total Gross': all_orders['sum_price_gross'].sum(),
        'Commission (Fees)': all_orders['fee_gross'].sum(),
        'Net Payout Expected': all_orders['payout'].sum(),
        'Order Count': len(all_orders)
    },
    'PAID Orders': {
        'Gross Sales': paid_orders['price_gross'].sum(),
        'Shipping': paid_orders['shipping_charges_gross'].sum(),
        'Total Gross': paid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': paid_orders['fee_gross'].sum(),
        'Net Payout Expected': paid_orders['payout'].sum(),
        'Order Count': len(paid_orders)
    },
    'UNPAID Orders': {
        'Gross Sales': unpaid_orders['price_gross'].sum(),
        'Shipping': unpaid_orders['shipping_charges_gross'].sum(),
        'Total Gross': unpaid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': unpaid_orders['fee_gross'].sum(),
        'Net Payout Expected': unpaid_orders['payout'].sum(),
        'Order Count': len(unpaid_orders)
    },
    'CANCELLED Orders': {
        'Gross Sales': cancelled_orders_list['price_gross'].sum(),
        'Shipping': cancelled_orders_list['shipping_charges_gross'].sum(),
        'Total Gross': cancelled_orders_list['sum_price_gross'].sum(),
        'Commission (Fees)': cancelled_orders_list['fee_gross'].sum(),
        'Net Payout Expected': cancelled_orders_list['payout'].sum(),
        'Order Count': len(cancelled_orders_list)
    }
}

summary_df = pd.DataFrame(summary).T
print("\n", summary_df.round(2))

# Actual amount received = sum of all Freigabe/Release transactions (negative values)
actual_received_freigabe = df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, RELEASE_KEYWORDS))]['Betrag'].sum()
avg_commission_rate = (all_orders['fee_gross'].sum() / all_orders['sum_price_gross'].sum() * 100) if all_orders['sum_price_gross'].sum() > 0 else 0

print(f"\n\nActual Payout RECEIVED (Freigabe/Release): {abs(actual_received_freigabe):.2f} EUR")
print(f"Expected Payout (Paid): {paid_orders['payout'].sum():.2f} EUR")
print(f"Expected Payout (Unpaid): {unpaid_orders['payout'].sum():.2f} EUR")
print(f"Expected Payout (Cancelled): {cancelled_orders_list['payout'].sum():.2f} EUR")
print(f"Average Commission: {avg_commission_rate:.2f}%")

In [7]:
# Categorize transactions (multi-language)
print("="*70)
print("CATEGORIZING TRANSACTIONS")
print("="*70)

# GMU categories
def categorize_gmu_transaction(row):
    text = str(row['booking_text'])
    if pd.isna(row['order_number']):
        if 'fee' in text.lower() or 'storno fee' in text.lower():
            return 'Fees - Cancelled Orders'
        elif contains_any(text, PAYOUT_KEYWORDS):
            return 'Payout Transfer'
        elif contains_any(text, BASE_FEE_KEYWORDS):
            return 'Grundgebühr (Base Fee)'
        else:
            return 'Other Non-Sales'
    else:
        if contains_any(text, RELEASE_KEYWORDS):
            return 'Sales - Released'
        else:
            return 'Sales - Other'

df_gmu['transaction_category'] = df_gmu.apply(categorize_gmu_transaction, axis=1)

# df_10000 categories
def categorize_payment_transaction(row):
    text = str(row['Buchungstext'])
    if contains_any(text, GOODS_RECEIPT_KEYWORDS):
        return 'Sales Income'
    elif contains_any(text, COMMISSION_KEYWORDS):
        return 'Commission/Fees'
    elif contains_any(text, RELEASE_KEYWORDS):
        return 'Sales Released (PAID)'
    elif contains_any(text, PAYOUT_KEYWORDS):
        return 'Payout Transfer'
    elif contains_any(text, BASE_FEE_KEYWORDS):
        return 'Grundgebühr (Base Fee)'
    elif contains_any(text, CANCEL_KEYWORDS):
        return 'Cancellation/Refund'
    else:
        return 'Other'

df_10000['transaction_category'] = df_10000.apply(categorize_payment_transaction, axis=1)

# Monthly breakdowns
gmu_category_monthly = df_gmu.pivot_table(
    values='payout',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)

# FIXED: Flip signs for payment categories to make them more readable
df10000_category_monthly = df_10000.pivot_table(
    values='Betrag',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)

# Flip sign for Sales Released (make it positive for readability)
if 'Sales Released (PAID)' in df10000_category_monthly.columns:
    df10000_category_monthly['Sales Released (PAID)'] = df10000_category_monthly['Sales Released (PAID)'].abs()

# Flip sign for Commission/Fees (make it positive for readability)
if 'Commission/Fees' in df10000_category_monthly.columns:
    df10000_category_monthly['Commission/Fees'] = df10000_category_monthly['Commission/Fees'].abs()

print("\n✓ Categories created (with corrected signs)")

CATEGORIZING TRANSACTIONS

✓ Categories created (with corrected signs)


In [ ]:
# Identify shipped orders and CANCELLED/RETURNED (including fees)
print("="*70)
print("SHIPPED & CANCELLED ORDERS ANALYSIS")
print("="*70)

# Shipped orders from GMU sales only (not the df_10000 additions)
shipped_orders = gmu_sales[
    (gmu_sales['payout'] > 0) & 
    (gmu_sales['booking_text'].apply(lambda x: contains_any(x, RELEASE_KEYWORDS)))
].copy()
shipped_orders['payment_status'] = shipped_orders['order_status']

# CANCELLED orders: Use order_status == 'CANCELLED'
cancelled_returned_orders = all_orders[all_orders['order_status'] == 'CANCELLED'].copy()

# Add cancelled fees (non-order transactions)
cancelled_fees = df_gmu[
    (df_gmu['order_number'].isna()) & 
    (df_gmu['booking_text'].str.lower().str.contains('fees for cancelled|storno', na=False))
].copy()

print(f"\nShipped orders: {len(shipped_orders)}")
print(f"Cancelled orders: {len(cancelled_returned_orders)}")
print(f"Cancelled fees (non-order): {len(cancelled_fees)}")
print(f"\nTotal cancelled/fees to report: {len(cancelled_returned_orders) + len(cancelled_fees)}")

In [9]:
# Payment timing analysis
print("="*70)
print("PAYMENT TIMING")
print("="*70)

paid_details = all_orders[all_orders['is_paid']].copy()

# Get dates from df_10000
freigabe_dates = df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, RELEASE_KEYWORDS))].copy()
freigabe_dates = freigabe_dates.sort_values('Datum').groupby('Bestellnummer')['Datum'].last()

wareneingang_dates = df_10000[df_10000['Buchungstext'].apply(lambda x: contains_any(x, GOODS_RECEIPT_KEYWORDS))].copy()
wareneingang_dates = wareneingang_dates.sort_values('Datum').groupby('Bestellnummer')['Datum'].first()

paid_details = paid_details.set_index('order_number')
paid_details['order_created_date'] = paid_details.index.map(wareneingang_dates)
paid_details['payment_received_date'] = paid_details.index.map(freigabe_dates)
paid_details = paid_details.reset_index()

paid_details['payment_delay_days'] = (paid_details['payment_received_date'] - paid_details['order_created_date']).dt.days

if len(paid_details[paid_details['payment_delay_days'].notna()]) > 0:
    print("\nPayment Delay (days from order to payment):")
    print(paid_details['payment_delay_days'].describe())
else:
    print("\nNo payment timing data available")

PAYMENT TIMING

Payment Delay (days from order to payment):
count    24.000000
mean     24.416667
std       5.523992
min      19.000000
25%      21.000000
50%      22.000000
75%      25.500000
max      38.000000
Name: payment_delay_days, dtype: float64


In [10]:
# Flag orders with missing/zero amounts
print("="*70)
print("CHECKING FOR MISSING AMOUNTS")
print("="*70)

# Only check GMU orders (not df_10000 additions)
gmu_sales_check = gmu_sales.copy()
gmu_sales_check['has_missing_price'] = (gmu_sales_check['price_gross'].isna()) | (gmu_sales_check['price_gross'] == 0)
gmu_sales_check['has_missing_payout'] = (gmu_sales_check['payout'].isna()) | (gmu_sales_check['payout'] == 0)
gmu_sales_check['has_missing_sum'] = (gmu_sales_check['sum_price_gross'].isna()) | (gmu_sales_check['sum_price_gross'] == 0)

problematic_orders = gmu_sales_check[
    gmu_sales_check['has_missing_price'] | 
    gmu_sales_check['has_missing_payout'] | 
    gmu_sales_check['has_missing_sum']
]

print(f"\nOrders with missing/zero amounts: {len(problematic_orders)}")
if len(problematic_orders) == 0:
    print("✓ All amounts are present")

CHECKING FOR MISSING AMOUNTS

Orders with missing/zero amounts: 0
✓ All amounts are present


In [ ]:
# Create All Orders Overview
print("="*70)
print("CREATING ALL ORDERS OVERVIEW")
print("="*70)

all_orders_overview = all_orders[[
    'booking_date', 'order_date', 'order_number', 'title_item',
    'price_gross', 'shipping_charges_gross', 'sum_price_gross',
    'fee_gross', 'payout', 'order_status', 'source'
]].copy()

# Add payment received date for paid orders
all_orders_overview = all_orders_overview.set_index('order_number')
all_orders_overview['payment_received_date'] = all_orders_overview.index.map(freigabe_dates)
all_orders_overview = all_orders_overview.reset_index()

# Add days since order
all_orders_overview['days_since_order'] = (pd.Timestamp.now() - all_orders_overview['booking_date']).dt.days

# Sort by order status (unpaid first, then cancelled, then paid) then by date
status_order = {'UNPAID': 1, 'CANCELLED': 2, 'PAID': 3}
all_orders_overview['status_sort'] = all_orders_overview['order_status'].map(status_order)
all_orders_overview = all_orders_overview.sort_values(['status_sort', 'booking_date'], ascending=[True, False])
all_orders_overview = all_orders_overview.drop('status_sort', axis=1)

print(f"\nTotal orders: {len(all_orders_overview)}")
print(f"PAID: {(all_orders_overview['order_status'] == 'PAID').sum()}")
print(f"UNPAID: {(all_orders_overview['order_status'] == 'UNPAID').sum()}")
print(f"CANCELLED: {(all_orders_overview['order_status'] == 'CANCELLED').sum()}")

In [ ]:
# Export to Excel - FINAL VERSION
print("="*70)
print("EXPORTING TO EXCEL")
print("="*70)

excel_filename = f'marketplace_financial_analysis_{COUNTRY.upper()}.xlsx'
writer = pd.ExcelWriter(excel_filename, engine='xlsxwriter')

# Tab 1: Summary
print("\n1. Summary...")
summary_df.to_excel(writer, sheet_name='1_Summary', startrow=0)
additional_info = pd.DataFrame({
    'Metric': [
        'Country',
        'Actual Payout Received (Freigabe/Release)',
        'Expected Payout (Paid)',
        'Difference',
        'Expected Payout (Unpaid/Pending)',
        'Expected Payout (Cancelled)',
        'Avg Commission Rate (%)',
        'Total Commissions',
        'Orders with Missing Amounts',
        'Unpaid Orders Count',
        'Cancelled Orders Count',
        'Shipped Orders',
        'Cancelled/Returned (incl fees)'
    ],
    'Value': [
        COUNTRY.upper(),
        abs(actual_received_freigabe),
        paid_orders['payout'].sum(),
        abs(actual_received_freigabe) - paid_orders['payout'].sum(),
        unpaid_orders['payout'].sum(),
        cancelled_orders_list['payout'].sum(),
        avg_commission_rate,
        all_orders['fee_gross'].sum(),
        len(problematic_orders),
        len(unpaid_orders),
        len(cancelled_orders_list),
        len(shipped_orders),
        len(cancelled_returned_orders) + len(cancelled_fees)
    ]
})
additional_info.to_excel(writer, sheet_name='1_Summary', startrow=len(summary_df)+3, index=False)

# Tab 2: ALL ORDERS OVERVIEW
print("2. All Orders Overview...")
all_orders_export = all_orders_overview[[
    'order_number', 'order_status', 'source', 'booking_date', 'order_date',
    'title_item', 'price_gross', 'sum_price_gross', 'fee_gross',
    'payout', 'payment_received_date', 'days_since_order'
]].copy()
all_orders_export.to_excel(writer, sheet_name='2_All_Orders_Overview', index=False)

# Tab 3: Unpaid Orders
print("3. Unpaid Orders...")
if len(unpaid_orders) > 0:
    unpaid_export = unpaid_orders[[
        'booking_date', 'order_date', 'order_number', 'source', 'title_item',
        'price_gross', 'sum_price_gross', 'fee_gross', 'payout'
    ]].copy().sort_values('booking_date', ascending=False)
    unpaid_export['days_since_order'] = (pd.Timestamp.now() - unpaid_export['booking_date']).dt.days
    unpaid_export.to_excel(writer, sheet_name='3_Unpaid_Orders_Detail', index=False)
else:
    pd.DataFrame({'Message': ['No unpaid orders']}).to_excel(writer, sheet_name='3_Unpaid_Orders_Detail', index=False)

# Tab 4: Paid Orders
print("4. Paid Orders...")
if len(paid_orders) > 0:
    paid_export = paid_orders[[
        'booking_date', 'order_date', 'order_number', 'title_item',
        'price_gross', 'sum_price_gross', 'fee_gross', 'payout'
    ]].copy().sort_values('booking_date', ascending=False)
    paid_export.to_excel(writer, sheet_name='4_Paid_Orders_Detail', index=False)
else:
    pd.DataFrame({'Message': ['No paid orders']}).to_excel(writer, sheet_name='4_Paid_Orders_Detail', index=False)

# Tab 5: Shipped Orders
print("5. Shipped Orders...")
if len(shipped_orders) > 0:
    shipped_export = shipped_orders[[
        'booking_date', 'order_date', 'order_number', 'title_item',
        'price_gross', 'sum_price_gross', 'fee_gross', 'payout',
        'payment_status'
    ]].copy().sort_values('booking_date', ascending=False)
    shipped_export.to_excel(writer, sheet_name='5_Shipped_Orders', index=False)
else:
    pd.DataFrame({'Message': ['No shipped orders']}).to_excel(writer, sheet_name='5_Shipped_Orders', index=False)

# Tab 6: Cancelled/Returned (including fees)
print("6. Cancelled/Returned...")
if len(cancelled_returned_orders) > 0 or len(cancelled_fees) > 0:
    # Combine cancelled orders and fees
    cancelled_combined = []
    
    # Add cancelled orders
    if len(cancelled_returned_orders) > 0:
        for _, row in cancelled_returned_orders.iterrows():
            cancelled_combined.append({
                'booking_date': row['booking_date'],
                'order_number': row['order_number'],
                'order_status': 'CANCELLED',
                'type': 'Cancelled Order',
                'title_item': row.get('title_item', ''),
                'price_gross': row['price_gross'],
                'sum_price_gross': row['sum_price_gross'],
                'fee_gross': row['fee_gross'],
                'payout': row['payout'],
                'booking_text': row.get('booking_text', '')
            })
    
    # Add cancelled fees
    if len(cancelled_fees) > 0:
        for _, row in cancelled_fees.iterrows():
            cancelled_combined.append({
                'booking_date': row['booking_date'],
                'order_number': 'N/A',
                'order_status': 'CANCELLED',
                'type': 'Cancelled Fee',
                'title_item': 'N/A',
                'price_gross': 0,
                'sum_price_gross': 0,
                'fee_gross': 0,
                'payout': row['payout'],
                'booking_text': row['booking_text']
            })
    
    cancelled_export = pd.DataFrame(cancelled_combined).sort_values('booking_date', ascending=False)
    cancelled_export.to_excel(writer, sheet_name='6_Cancelled_Returned', index=False)
else:
    pd.DataFrame({'Message': ['No cancelled/returned']}).to_excel(writer, sheet_name='6_Cancelled_Returned', index=False)

# Tab 7: Payment Timing
print("7. Payment Timing...")
if len(paid_details) > 0:
    timing_export = paid_details[[
        'order_number', 'booking_date', 'order_created_date', 'payment_received_date',
        'payment_delay_days', 'price_gross', 'payout'
    ]].copy().sort_values('payment_received_date', ascending=False)
    timing_export.to_excel(writer, sheet_name='7_Payment_Timing', index=False)
else:
    pd.DataFrame({'Message': ['No payment timing data']}).to_excel(writer, sheet_name='7_Payment_Timing', index=False)

# Tab 8: Monthly GMU Categories
print("8. Monthly GMU Categories...")
gmu_cat_export = gmu_category_monthly.copy()
gmu_cat_export.index = gmu_cat_export.index.astype(str)
gmu_cat_export.to_excel(writer, sheet_name='8_Monthly_GMU_Categories')

# Tab 9: Monthly Payment Categories (CORRECTED SIGNS)
print("9. Monthly Payment Categories...")
df10k_cat_export = df10000_category_monthly.copy()
df10k_cat_export.index = df10k_cat_export.index.astype(str)
df10k_cat_export.to_excel(writer, sheet_name='9_Monthly_Payment_Categories')

# Tab 10: Missing Amounts ERROR
print("10. Missing Amounts...")
if len(problematic_orders) > 0:
    prob_export = problematic_orders[[
        'booking_date', 'order_number', 'booking_text', 'price_gross',
        'sum_price_gross', 'payout', 'has_missing_price',
        'has_missing_payout', 'has_missing_sum'
    ]].copy().sort_values('booking_date', ascending=False)
    prob_export.to_excel(writer, sheet_name='10_Missing_Amounts_ERROR', index=False)
else:
    pd.DataFrame({'Message': ['No missing amounts']}).to_excel(writer, sheet_name='10_Missing_Amounts_ERROR', index=False)

# Tab 11: All GMU Transactions
print("11. All GMU Transactions...")
gmu_export = df_gmu.copy()
gmu_export['month'] = gmu_export['month'].astype(str)
gmu_export.to_excel(writer, sheet_name='11_All_GMU_Transactions', index=False)

# Tab 12: All Payments (df_10000)
print("12. All Payments Received...")
df10k_export = df_10000.copy()
df10k_export['month'] = df10k_export['month'].astype(str)
df10k_export = df10k_export.sort_values('Datum', ascending=False)
df10k_export.to_excel(writer, sheet_name='12_All_Payments_Received', index=False)

writer.close()

print(f"\n✓ Excel file created: {excel_filename}")
print(f"\n✅ ANALYSIS COMPLETE FOR {COUNTRY.upper()}!")
print("\nTo analyze another country, change the COUNTRY variable in Cell 2 and run all cells again.")